In [ ]:
# Imports
import scipy.io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, interact_manual, widgets

#### Load csv with pandas and display it

In [ ]:
settings_DF=pd.read_csv('/Volumes/FBK-NAS/01_Tool_WearDatasets/00_NASA_Milling/settings.csv')
# the ID column of settings_DF is unnamed, so we rename it
#settings_DF.drop(columns={'Unnamed: 0'}, inplace=True)
# load signals and merge them, as they have all the same shape
signal_DFs=[]
for signal in ['smcAC', 'smcDC', 'vib_table', 'vib_spindle', 'AE_table', 'AE_spindle']:
    signal_DFs.append(pd.read_csv('/Volumes/FBK-NAS/01_Tool_WearDatasets/00_NASA_Milling/'+signal+'.csv'))
    print(signal_DFs[-1].shape)
# merge the signals
merged_DF=pd.concat(signal_DFs, axis=1)
# remove duplicate ID column
merged_DF=merged_DF.loc[:,~merged_DF.columns.duplicated()]
merged_DF.head(10)

### Remove outliers 
If we go through the cuts, we spot two outliers, that should be erased for cut_no: 17 and 94. These would otherwise moderate our statistics.

In [ ]:
settings_DF.drop(settings_DF.loc[settings_DF['cut_no'].isin([17,94])].index, inplace=True)
merged_DF.drop(merged_DF.loc[merged_DF['cut_no'].isin([17,94])].index, inplace=True)

In [ ]:
# Now we built a normalized wear column
settings_DF['wear_norm'] = (settings_DF['VB'] - settings_DF['VB'].min()) / (settings_DF['VB'].max() - settings_DF['VB'].min())

In [ ]:
#Visualize the wear ('VB') over time ('run') for each case ('case')
import seaborn as sns
plt.figure(figsize=(12, 6))
sns.lineplot(data=settings_DF, x='run', y='wear_norm', hue='case', marker='o')
plt.title('Tool Wear Over Time for Each Case')
plt.xlabel('Cut Number')
plt.ylabel('Normalized Tool Wear (1 = No Wear, 0 = Max Wear)')
plt.legend(title='Case ID', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# Now lets build a homogenized data file for comparison with the other datasets
# We rename VB to wear, case to CaseID, run to run
settings_DF = settings_DF.rename(columns={'VB': 'wear', 'case': 'CaseID', 'run': 'run','cut_no':'ExperimentIndex'})
merged_DF.rename(columns={'cut_no': 'ExperimentIndex'}, inplace=True)
settings_DF.head()
# Then we merge the settings_DF with the merged_DF on the cut_no column
# First, we aggregate the signals for each cut_no into a numpy array.
signal_cols = ['smcAC', 'smcDC', 'vib_table', 'vib_spindle', 'AE_table', 'AE_spindle']
signals_agg = merged_DF.groupby('ExperimentIndex')[signal_cols].agg(list).reset_index()
for col in signal_cols:
    signals_agg[col] = signals_agg[col].apply(np.array)

# From settings_DF, we only need case, run, and cut_no for the new dataframe.
settings_subset = settings_DF[['CaseID', 'run', 'ExperimentIndex', 'wear_norm']]

# Now, we merge the two dataframes on 'cut_no'
nasa_df = pd.merge(settings_subset, signals_agg, on='ExperimentIndex')
nasa_df.head()


In [ ]:
# save as parquet file
nasa_df.to_parquet('/Volumes/FBK-NAS/01_Tool_WearDatasets/00_NASA_Milling/nasa_milling_normalized.parquet')